# 03 -- Interpolate Transportation Tomorrow Survey data onto ADAs

Transfers the households-without-a-vehicle share (`hh_no_veh_pct`, the "Households with No Vehicle" map layer) from the TTS 2022 zones onto the ADA polygons by **area-weighted interpolation** (`tobler`).



---

| | |
|---|---|
| **Inputs** | `data/census/ada-wide/toronto-ada-wide.geojson` (ADA attributes, from step 02)<br>`data/census/tts/tts2022.geojson` (TTS zone data, download) |
| **Outputs** | `data/census/ada-wide/ada-wide-tts.geojson|gpkg` -- the ADA layer plus `hh_no_veh_pct` (intermediate; input to step 04) |
| **Run after** | `02_prepare_ada_wide.ipynb`. Next: `04_interpolate_noc_naics.ipynb`. |

Requires `pip install tobler`. The last cell prints quintile breaks for the legend in `src/lib/maps/tacLayerConfig.js`.


In [22]:
import geopandas as gpd
from tobler.area_weighted import area_interpolate

from pathlib import Path

# Repo root = nearest parent folder containing package.json, so this notebook runs from any depth.
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "package.json").exists())
DATA = ROOT / "data"                 # raw + intermediate files
SRC_DATA = ROOT / "src" / "data"     # final outputs consumed directly by the web app

## 1. Load

In [23]:
# TTS data
tts = gpd.read_file(str(DATA / "census/tts/tts2022.geojson"))

# ADA wide data
ada = gpd.read_file(str(DATA / "census/ada-wide/toronto-ada-wide.geojson"))

## 2. Areal interpolation

Projects both layers to UTM 17N, repairs geometries and area-weights `Perc_No_Veh` onto each ADA as `hh_no_veh_pct`.

In [24]:
# Areal interpolation

# Reproject to UTM 17N
tts = tts.to_crs(epsg=32617)
ada = ada.to_crs(epsg=32617)

# Fix invalid geometries
tts["geometry"] = tts.geometry.make_valid()
ada["geometry"] = ada.geometry.make_valid()

# Drop tts NA
tts_valid = tts.dropna(subset=["Perc_No_Veh", "Pop_Dens", "Veh_Per_Hhld"], how="all").copy()

# Interpolate
result = area_interpolate(
    source_df=tts_valid,
    target_df=ada,
    intensive_variables=["Perc_No_Veh", "Pop_Dens", "Veh_Per_Hhld"],
)

# Attach to ada
ada["hh_no_veh_pct"] = result["Perc_No_Veh"].values.round(1)
# ada["pop_dens_interp"] = result["Pop_Dens"].values
# ada["veh_per_hhld_interp"] = result["Veh_Per_Hhld"].values

# Project to WGS84 (EPSG:4326)
ada = ada.to_crs(epsg=4326)

## 3. Write outputs

In [25]:
# Write files
ada.to_file(
    str(DATA / "census/ada-wide/ada-wide-tts.geojson"), 
    driver="GeoJSON"
)

ada.to_file(
    str(DATA / "census/ada-wide/ada-wide-tts.gpkg"),
    layer="ada",
    driver="GPKG"
)

## 4. Legend breaks

Quintile breaks to paste into `tacLayerConfig.js` if this layer is added.

In [28]:
# Print quantile breaks for tacLayerConfig.js
quantiles = ada["hh_no_veh_pct"].quantile([0.2, 0.4, 0.6, 0.8])
print(quantiles)

0.2     9.74
0.4    15.20
0.6    22.28
0.8    35.92
Name: hh_no_veh_pct, dtype: float64
